In [9]:
RUN_TARGET = "colab"  # "colab" | "local"
# adjust OVERWRITE_EXISTING 

## Setup Instructions

Notebook 06 generates residue-level probe rows for supported active model families. It is registry-based rather than fully model-agnostic: unknown checkpoints are skipped, and retired top-1-unfrozen baseline checkpoints are intentionally excluded. Protein-level classification metrics are not generated here; notebook 07 loads them later from saved JSON artifacts.

### Typical workflow
1. Train or update a supported checkpoint in notebook 03, 04, or 05.
2. Run this notebook to generate or refresh saved probe-row CSVs in `results/probing/rows/`.
3. Open `07_compare_all_model_probes.ipynb` to build paper figures and tables from those saved artifacts.


In [10]:
if RUN_TARGET == "colab":
    import importlib.metadata as _md
    import subprocess as _sp
    import sys as _sys

    _required = {
        "numpy": "1.26.4",
        "pandas": "2.3.2",
        "scikit-learn": "1.8.0",
        "transformers": "4.48.1",
        "huggingface-hub": "0.36.2",
        "seaborn": "0.13.2",
        "captum": None,
    }

    def _version_matches(name: str, expected: str | None) -> bool:
        try:
            installed = _md.version(name)
            return installed == expected if expected is not None else True
        except _md.PackageNotFoundError:
            return False

    _missing_or_mismatched = [
        f"{name}=={version}" if version is not None else name
        for name, version in _required.items()
        if not _version_matches(name, version)
    ]

    if _missing_or_mismatched:
        print("Installing:", ", ".join(_missing_or_mismatched))
        _sp.run([
            _sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--upgrade",
            *_missing_or_mismatched,
        ], check=True)
        raise SystemExit(
            "Colab environment updated. Restart the runtime once, then rerun the notebook from the top."
        )
    else:
        print("Colab environment already compatible. No reinstall needed.")
else:
    print("Local environment detected. Skipping Colab setup.")


Colab environment already compatible. No reinstall needed.


In [11]:
if RUN_TARGET == "colab":
    from google.colab import drive as _drive
    from pathlib import Path

    _drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = Path("/content/drive/MyDrive/XAllergen2.0")
    DRIVE_MODELS = DRIVE_ROOT / "models"
    DRIVE_RESULTS = DRIVE_ROOT / "results"
    print(f"Google Drive mounted: {DRIVE_ROOT}")
else:
    print("Local run: skipping Google Drive mount.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted: /content/drive/MyDrive/XAllergen2.0


# 06 - Registry-Driven Probe Row Generation

This notebook centralizes residue-level probe generation for the supported active model registry. It is semi-automatic over known model families, not a generic sweep over arbitrary checkpoint names. Unknown checkpoints are skipped, retired top-1-unfrozen baselines are ignored, and supported MTL probe rows are generated against the frozen baseline reference.


In [12]:
import sys
import urllib.request
from pathlib import Path

RAW = "https://raw.githubusercontent.com/Jeffateth/XAllergen2.0/main"


def add_src_to_path(project_root: Path, prepend: bool = True) -> bool:
    src_dir = project_root / "src"
    package_dir = src_dir / "xallergen"
    if package_dir.exists():
        if str(src_dir) in sys.path:
            sys.path.remove(str(src_dir))
        if prepend:
            sys.path.insert(0, str(src_dir))
        else:
            sys.path.append(str(src_dir))
        return True
    return False


def ensure_colab_xallergen_package(runtime_root: Path, refresh: bool = False) -> Path:
    src_dir = runtime_root / "src"
    package_dir = src_dir / "xallergen"
    package_dir.mkdir(parents=True, exist_ok=True)
    for module_name in [
        "__init__.py",
        "baseline_notebook_utils.py",
        "deep_plant_allergy_utils.py",
        "mtl_epitope_notebook_utils.py",
        "plotting_paper_figures.py",
    ]:
        target = package_dir / module_name
        if refresh or not target.exists():
            urllib.request.urlretrieve(f"{RAW}/src/xallergen/{module_name}", target)
            print(f"Downloaded: {module_name}")
    if str(src_dir) in sys.path:
        sys.path.remove(str(src_dir))
    sys.path.insert(0, str(src_dir))
    return src_dir


import importlib
import json

if RUN_TARGET == "colab":
    candidate_roots = []
    if "DRIVE_ROOT" in globals():
        candidate_roots.append(DRIVE_ROOT)
    candidate_roots.extend([
        Path("/content/drive/MyDrive/XAllergen2.0"),
        Path("/content/XAllergen2.0"),
    ])

    RUNTIME_ROOT = None
    for _root in candidate_roots:
        if add_src_to_path(_root):
            RUNTIME_ROOT = _root
            print(f"Using xallergen source from {_root / 'src'}")
            break

    if RUNTIME_ROOT is None:
        RUNTIME_ROOT = Path("/content/XAllergen2.0")
        _src_dir = ensure_colab_xallergen_package(RUNTIME_ROOT, refresh=True)
        print(f"Bootstrapped xallergen source into {_src_dir}")
    elif RUNTIME_ROOT == Path("/content/XAllergen2.0"):
        _src_dir = ensure_colab_xallergen_package(RUNTIME_ROOT, refresh=True)
        print(f"Refreshed xallergen source in {_src_dir}")

    if str(RUNTIME_ROOT) in sys.path:
        sys.path.remove(str(RUNTIME_ROOT))
    sys.path.insert(0, str(RUNTIME_ROOT))
else:
    for _candidate in [Path.cwd(), *Path.cwd().parents]:
        if add_src_to_path(_candidate):
            break

import xallergen.baseline_notebook_utils as baseline_notebook_utils
import xallergen.deep_plant_allergy_utils as deep_plant_allergy_utils
import xallergen.mtl_epitope_notebook_utils as mtl_epitope_notebook_utils
import xallergen.plotting_paper_figures as plotting_paper_figures

importlib.reload(baseline_notebook_utils)
importlib.reload(deep_plant_allergy_utils)
importlib.reload(mtl_epitope_notebook_utils)
importlib.reload(plotting_paper_figures)

from xallergen.baseline_notebook_utils import (
    DROPOUT,
    HF_MODEL_NAME,
    HIDDEN_DIM,
    IG_STEPS,
    build_tokenizer,
    configure_matplotlib_cache,
    detect_device,
    find_project_root,
    load_baseline_checkpoint,
    load_mtl_checkpoint,
    prepare_baseline_probe_frame,
    print_runtime_context,
    run_baseline_probe_suite,
)
from xallergen.deep_plant_allergy_utils import (
    build_embedding_model,
    build_tokenizer as build_deep_plant_tokenizer,
    load_deep_plant_allergy_checkpoint,
    run_deep_plant_probe_suite,
)
from xallergen.mtl_epitope_notebook_utils import (
    get_retired_checkpoint_names,
    get_supported_probe_model_registry,
    run_probe_suite,
    summarize_probe_outputs,
)
from xallergen.plotting_paper_figures import (
    build_output_paths_for_supported_mtl,
    save_registry_probe_summary,
)

if RUN_TARGET != "colab":
    configure_matplotlib_cache(Path.cwd())

import pandas as pd
import torch


Using xallergen source from /content/XAllergen2.0/src
Downloaded: __init__.py
Downloaded: baseline_notebook_utils.py
Downloaded: deep_plant_allergy_utils.py
Downloaded: mtl_epitope_notebook_utils.py
Downloaded: plotting_paper_figures.py
Refreshed xallergen source in /content/XAllergen2.0/src


In [13]:
if RUN_TARGET == "colab":
    DRIVE_PROJECT_ROOT = DRIVE_ROOT if "DRIVE_ROOT" in globals() else Path("/content/drive/MyDrive/XAllergen2.0")
    PROJECT_ROOT = DRIVE_PROJECT_ROOT if DRIVE_PROJECT_ROOT.exists() else RUNTIME_ROOT
    MODELS_DIR = DRIVE_MODELS if "DRIVE_MODELS" in globals() and DRIVE_MODELS.exists() else PROJECT_ROOT / "models"
    RESULTS_DIR = DRIVE_RESULTS if "DRIVE_RESULTS" in globals() and DRIVE_RESULTS.exists() else PROJECT_ROOT / "results"
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Project root: {PROJECT_ROOT}")
    print(f"Using models dir: {MODELS_DIR}")
    print(f"Using results dir: {RESULTS_DIR}")
    print(f"Device: {DEVICE}")
else:
    PROJECT_ROOT = find_project_root(Path.cwd())
    DEVICE = detect_device()
    print_runtime_context(DEVICE, PROJECT_ROOT)
    MODELS_DIR = PROJECT_ROOT / "models"
    RESULTS_DIR = PROJECT_ROOT / "results"

DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
for _output_subdir in [
    RESULTS_DIR / "classification",
    RESULTS_DIR / "probing" / "rows",
    RESULTS_DIR / "probing" / "summaries",
    RESULTS_DIR / "figures" / "diagnostics",
]:
    _output_subdir.mkdir(parents=True, exist_ok=True)

POSITIVES_CSV = DATA_DIR / "positives_splitB.csv"
tokenizer = build_tokenizer(HF_MODEL_NAME)
epitope_probe_df = prepare_baseline_probe_frame(POSITIVES_CSV)
print(f"Probe proteins in splitB: {len(epitope_probe_df)}")


Project root: /content/drive/MyDrive/XAllergen2.0
Using models dir: /content/drive/MyDrive/XAllergen2.0/models
Using results dir: /content/drive/MyDrive/XAllergen2.0/results
Device: cuda
Probe proteins in splitB: 61


## Generation Parameters

- `OVERWRITE_EXISTING` keeps existing probe artifacts when set to `False`.
- `N_RANDOM_DRAWS` controls the random-null estimate written into the probe rows.
- `IG_INTERNAL_BATCH_SIZE` controls the memory/speed tradeoff for Captum IG and SmoothGrad-IG.
- `INCLUDE_INTEGRATED_GRADIENTS`, `INCLUDE_GRADIENT_X_INPUT`, and `INCLUDE_SMOOTHGRAD_IG` let you include or exclude the expensive gradient-based methods from repair runs.
- `REPAIR_FAMILY_KEYS` and `REPAIR_METHODS` drive the standalone repair cell so you can regenerate only missing or failed methods without overwriting successful probe rows.


In [14]:
OVERWRITE_EXISTING = False
N_RANDOM_DRAWS = 100
IG_INTERNAL_BATCH_SIZE = 64
SMOOTHGRAD_IG_SAMPLES = 10
SMOOTHGRAD_IG_NOISE_STD = 0.05

INCLUDE_INTEGRATED_GRADIENTS = True
INCLUDE_GRADIENT_X_INPUT = True
INCLUDE_SMOOTHGRAD_IG = True

REPAIR_FAMILY_KEYS = ["baseline", "mtl_frozen", "mtl_top1_unfrozen"]
REPAIR_METHODS = ["integrated_gradients"]


## Supported Registry

The supported-model registry defines which active model families notebook 06 knows how to probe. Unknown checkpoints are skipped rather than inferred automatically. Optional `MTL ESM-2 top-1` is supplementary only, and retired top-1-unfrozen baseline checkpoints are intentionally excluded.


In [15]:
retired_checkpoint_names = set(get_retired_checkpoint_names())
supported_registry = get_supported_probe_model_registry(
    models_dir=MODELS_DIR,
    results_dir=RESULTS_DIR,
    include_supplementary=True,
)
supported_lookup = {spec["checkpoint_name"]: spec for spec in supported_registry}
registry_df = pd.DataFrame(
    [
        {
            "family_key": spec["family_key"],
            "display_label": spec["display_label"],
            "checkpoint_name": spec["checkpoint_name"],
            "checkpoint_exists": spec["checkpoint_exists"],
            "probe_rows_path": str(spec["probe_rows_path"]),
            "summary_path": str(spec["summary_path"]),
            "supplementary_only": spec["supplementary_only"],
            "supported_methods": ", ".join(spec["supported_methods"]),
        }
        for spec in supported_registry
    ]
)
registry_df


,family_key,display_label,checkpoint_name,checkpoint_exists,probe_rows_path,summary_path,supplementary_only,supported_methods
0,baseline,Frozen ESM-2,baseline_frozen_esm2.pt,True,/content/drive/MyDrive/XAllergen2.0/results/pr...,/content/drive/MyDrive/XAllergen2.0/results/pr...,False,"attention_weights, integrated_gradients, gradi..."
1,mtl_frozen,MTL ESM-2,mtl_frozen_esm2_epitope.pt,True,/content/drive/MyDrive/XAllergen2.0/results/pr...,/content/drive/MyDrive/XAllergen2.0/results/pr...,False,"residue_head, attention_weights, integrated_gr..."
2,deep_plant_benchmark,DeepPlantAllergy,deep_plant_allergy_benchmark.pt,True,/content/drive/MyDrive/XAllergen2.0/results/pr...,/content/drive/MyDrive/XAllergen2.0/results/pr...,False,"attention_weights, integrated_gradients, rando..."
3,mtl_top1_unfrozen,MTL ESM-2 top-1,mtl_top1_unfrozen_esm2_epitope.pt,True,/content/drive/MyDrive/XAllergen2.0/results/pr...,/content/drive/MyDrive/XAllergen2.0/results/pr...,True,"residue_head, attention_weights, integrated_gr..."


## Probe Generation

The frozen baseline probe rows are generated first and then reused by supported MTL families. DeepPlantAllergy and supported MTL checkpoints are generated only if they match the registry. Retired baseline checkpoints are skipped explicitly, and unsupported checkpoint names are recorded as unknown.


## Standalone Repair Cell

Use the next cell when a gradient-based method fails after other probe methods have already finished.
It patches the live runtime, recomputes only the selected methods in `REPAIR_METHODS`, and merges them back into the existing CSVs so successful rows are preserved.


In [ ]:
import contextlib

from captum.attr import IntegratedGradients


def cudnn_backward_safe_context(device: str):
    if str(device).startswith("cuda"):
        return torch.backends.cudnn.flags(enabled=False)
    return contextlib.nullcontext()


def enabled_gradient_methods() -> set[str]:
    enabled = set()
    if INCLUDE_INTEGRATED_GRADIENTS:
        enabled.add("integrated_gradients")
    if INCLUDE_GRADIENT_X_INPUT:
        enabled.add("gradient_x_input")
    if INCLUDE_SMOOTHGRAD_IG:
        enabled.add("smoothgrad_ig")
    return enabled


def patch_live_attribution_helpers() -> None:
    def safe_compute_integrated_gradients(
        model,
        tokenizer,
        sequence: str,
        device: str,
        steps: int = baseline_notebook_utils.IG_STEPS,
        normalize: bool = False,
        internal_batch_size: int | None = 1,
    ) -> np.ndarray:
        encodings = baseline_notebook_utils.tokenize_sequence(tokenizer, sequence, device)
        attention_mask = encodings["attention_mask"]
        input_embeds = model.backbone.get_input_embeddings()(encodings["input_ids"]).detach()
        baseline = torch.zeros_like(input_embeds)

        def ig_forward(inputs_embeds: torch.Tensor) -> torch.Tensor:
            return model.forward_from_inputs_embeds(inputs_embeds, attention_mask)["logits"]

        with cudnn_backward_safe_context(device):
            attributions = IntegratedGradients(ig_forward).attribute(
                inputs=input_embeds,
                baselines=baseline,
                n_steps=steps,
                internal_batch_size=internal_batch_size,
            )
        importance = attributions.abs().sum(dim=-1).squeeze(0).detach().cpu().numpy()
        valid_length = int(attention_mask.sum().item())
        importance = importance[:valid_length]
        return baseline_notebook_utils.normalize_scores(importance) if normalize else importance

    def safe_compute_gradient_x_input_scores(model, tokenizer, sequence: str, device: str) -> np.ndarray:
        model.eval()
        encodings = baseline_notebook_utils.tokenize_sequence(tokenizer, sequence, device)
        attention_mask = encodings["attention_mask"]
        model.zero_grad(set_to_none=True)
        input_embeds = model.backbone.get_input_embeddings()(encodings["input_ids"]).detach()
        input_embeds.requires_grad_(True)
        with cudnn_backward_safe_context(device):
            logits = model.forward_from_inputs_embeds(input_embeds, attention_mask)["logits"]
            gradients = torch.autograd.grad(
                outputs=logits.sum(),
                inputs=input_embeds,
                retain_graph=False,
                create_graph=False,
                only_inputs=True,
            )[0]
        scores = (input_embeds * gradients).sum(dim=-1).abs().squeeze(0)
        valid_length = int(attention_mask.sum().item())
        scores = scores[:valid_length].detach().cpu().numpy().astype(np.float32)
        model.zero_grad(set_to_none=True)
        return scores

    def safe_compute_smoothgrad_ig_scores(
        model,
        tokenizer,
        sequence: str,
        device: str,
        steps: int,
        n_samples: int = 10,
        noise_std: float = 0.05,
        internal_batch_size: int = 10,
    ) -> np.ndarray:
        model.eval()
        encodings = baseline_notebook_utils.tokenize_sequence(tokenizer, sequence, device)
        attention_mask = encodings["attention_mask"]
        base_embeds = model.backbone.get_input_embeddings()(encodings["input_ids"]).detach()
        baseline = torch.zeros_like(base_embeds)
        noise_scale = float(noise_std) * float(base_embeds.detach().std().item())
        total_scores = np.zeros(int(attention_mask.sum().item()), dtype=np.float64)

        def ig_forward(inputs_embeds: torch.Tensor) -> torch.Tensor:
            return model.forward_from_inputs_embeds(inputs_embeds, attention_mask)["logits"]

        ig = IntegratedGradients(ig_forward)
        for _ in range(n_samples):
            model.zero_grad(set_to_none=True)
            noisy_embeds = base_embeds + torch.randn_like(base_embeds) * noise_scale if noise_scale > 0 else base_embeds
            with cudnn_backward_safe_context(device):
                attributions = ig.attribute(
                    inputs=noisy_embeds.detach(),
                    baselines=baseline,
                    n_steps=steps,
                    internal_batch_size=internal_batch_size,
                )
            scores = attributions.abs().sum(dim=-1).squeeze(0).detach().cpu().numpy()
            total_scores += scores[: total_scores.shape[0]]
        model.zero_grad(set_to_none=True)
        return (total_scores / n_samples).astype(np.float32)

    def safe_deep_plant_compute_integrated_gradients(
        model,
        embedding_model,
        tokenizer,
        sequence: str,
        device: str,
        steps: int = deep_plant_allergy_utils.IG_STEPS,
        normalize: bool = False,
        internal_batch_size: int | None = 1,
    ) -> np.ndarray:
        residue_embeddings = deep_plant_allergy_utils.compute_residue_embeddings(embedding_model, tokenizer, sequence, device)
        baseline = torch.zeros_like(residue_embeddings)

        def ig_forward(inputs_embeds: torch.Tensor) -> torch.Tensor:
            return model(inputs_embeds).squeeze(-1)

        with cudnn_backward_safe_context(device):
            attributions = IntegratedGradients(ig_forward).attribute(
                inputs=residue_embeddings,
                baselines=baseline,
                n_steps=steps,
                internal_batch_size=internal_batch_size,
            )
        importance = attributions.abs().sum(dim=-1).squeeze(0).detach().cpu().numpy()
        return deep_plant_allergy_utils.normalize_scores(importance) if normalize else importance

    baseline_notebook_utils.compute_integrated_gradients = safe_compute_integrated_gradients
    baseline_notebook_utils.compute_gradient_x_input_scores = safe_compute_gradient_x_input_scores
    baseline_notebook_utils.compute_smoothgrad_ig_scores = safe_compute_smoothgrad_ig_scores
    mtl_epitope_notebook_utils.compute_integrated_gradients = safe_compute_integrated_gradients
    mtl_epitope_notebook_utils.compute_gradient_x_input_scores = safe_compute_gradient_x_input_scores
    mtl_epitope_notebook_utils.compute_smoothgrad_ig_scores = safe_compute_smoothgrad_ig_scores
    deep_plant_allergy_utils.compute_integrated_gradients = safe_deep_plant_compute_integrated_gradients
    print("Patched live attribution helpers for CUDA-safe backward passes.")


def selected_methods_for_spec(spec: dict) -> list[str]:
    methods = []
    for method in spec["supported_methods"]:
        if method == "integrated_gradients" and not INCLUDE_INTEGRATED_GRADIENTS:
            continue
        if method == "gradient_x_input" and not INCLUDE_GRADIENT_X_INPUT:
            continue
        if method == "smoothgrad_ig" and not INCLUDE_SMOOTHGRAD_IG:
            continue
        methods.append(method)
    return methods


def merge_probe_rows(existing_df: pd.DataFrame, new_df: pd.DataFrame, replaced_methods: set[str]) -> pd.DataFrame:
    existing_df = existing_df.copy() if existing_df is not None and not existing_df.empty else pd.DataFrame()
    new_df = new_df.copy() if new_df is not None and not new_df.empty else pd.DataFrame()
    if existing_df.empty:
        merged = new_df
    else:
        preserved = existing_df.loc[~existing_df["method"].astype(str).isin(replaced_methods)].copy()
        merged = pd.concat([preserved, new_df], ignore_index=True)
    if merged.empty:
        return merged
    merged = mtl_epitope_notebook_utils.ensure_label_variant_column(merged)
    mtl_epitope_notebook_utils.validate_unique_probe_rows(merged)
    return merged


def recompute_baseline_gradient_methods(spec: dict, methods: set[str]) -> pd.DataFrame:
    if not methods:
        return pd.DataFrame()
    baseline_model, _ = load_baseline_checkpoint(spec["checkpoint_path"], DEVICE)
    rng = np.random.default_rng(baseline_notebook_utils.RANDOM_STATE)
    rows = []
    desc = f"Repairing [{spec['display_label']}]"
    for _, row in tqdm(epitope_probe_df.iterrows(), total=len(epitope_probe_df), desc=desc):
        sequence = row["sequence"]
        labels = row["epitope_label"]
        base = {
            "accession": str(row["accession"]),
            "seq_len": row["seq_len"],
            "epitope_density": row["epitope_density"],
            "n_epitope_residues": row["n_epitope_residues"],
            "model_family": spec["display_label"],
        }
        if "integrated_gradients" in methods:
            scores = baseline_notebook_utils.compute_integrated_gradients(
                baseline_model, tokenizer, sequence, DEVICE, steps=IG_STEPS, normalize=False, internal_batch_size=IG_INTERNAL_BATCH_SIZE,
            )
            rows.append({
                **base,
                "method": "integrated_gradients",
                "ig_scores_json": baseline_notebook_utils.serialize_score_array(scores),
                **baseline_notebook_utils.compute_probe_metrics(labels, scores),
            })
        if "gradient_x_input" in methods:
            scores = baseline_notebook_utils.compute_gradient_x_input_scores(baseline_model, tokenizer, sequence, DEVICE)
            rows.append({
                **base,
                "method": "gradient_x_input",
                "gradient_x_input_scores_json": baseline_notebook_utils.serialize_score_array(scores),
                **baseline_notebook_utils.compute_probe_metrics(labels, scores),
            })
        if "smoothgrad_ig" in methods:
            scores = baseline_notebook_utils.compute_smoothgrad_ig_scores(
                baseline_model, tokenizer, sequence, DEVICE, steps=IG_STEPS, n_samples=SMOOTHGRAD_IG_SAMPLES, noise_std=SMOOTHGRAD_IG_NOISE_STD, internal_batch_size=IG_INTERNAL_BATCH_SIZE,
            )
            rows.append({
                **base,
                "method": "smoothgrad_ig",
                "smoothgrad_ig_scores_json": baseline_notebook_utils.serialize_score_array(scores),
                **baseline_notebook_utils.compute_probe_metrics(labels, scores),
            })
    return pd.DataFrame(rows)


def recompute_mtl_gradient_methods(spec: dict, methods: set[str]):
    if not methods:
        return pd.DataFrame(), pd.DataFrame()
    output_paths = build_output_paths_for_supported_mtl(
        family_key=spec["family_key"],
        display_label=spec["display_label"],
        models_dir=MODELS_DIR,
        results_dir=RESULTS_DIR,
        baseline_checkpoint_path=baseline_spec["checkpoint_path"],
        baseline_summary_path=baseline_spec["summary_path"],
    )
    model, _ = load_mtl_checkpoint(spec["checkpoint_path"], DEVICE, model_name=HF_MODEL_NAME, hidden_dim=HIDDEN_DIM, dropout=DROPOUT)
    baseline_model, _ = load_baseline_checkpoint(baseline_spec["checkpoint_path"], DEVICE, model_name=HF_MODEL_NAME, hidden_dim=HIDDEN_DIM, dropout=DROPOUT)
    rng = np.random.default_rng(mtl_epitope_notebook_utils.RANDOM_STATE)
    probe_rows = []
    baseline_rows = []
    desc = f"Repairing splitB [{spec['display_label']}]"
    for _, row in tqdm(epitope_probe_df.iterrows(), total=len(epitope_probe_df), desc=desc):
        sequence = row["sequence"]
        labels = row["epitope_label"]
        base = {
            "accession": str(row["accession"]),
            "seq_len": int(row["seq_len"]),
            "epitope_density": float(row["epitope_density"]),
            "n_epitope_residues": int(row["n_epitope_residues"]),
        }
        if "integrated_gradients" in methods:
            mtl_scores = baseline_notebook_utils.compute_integrated_gradients(model, tokenizer, sequence, DEVICE, steps=IG_STEPS, normalize=False, internal_batch_size=IG_INTERNAL_BATCH_SIZE)
            probe_rows.extend(mtl_epitope_notebook_utils.build_probe_rows_with_label_scrambling(base, output_paths.mtl_family_label, "integrated_gradients", labels, mtl_scores, rng, serialize_scores=True, score_column="ig_scores_json"))
            baseline_scores = baseline_notebook_utils.compute_integrated_gradients(baseline_model, tokenizer, sequence, DEVICE, steps=IG_STEPS, normalize=False, internal_batch_size=IG_INTERNAL_BATCH_SIZE)
            baseline_rows.extend(mtl_epitope_notebook_utils.build_probe_rows_with_label_scrambling(base, output_paths.baseline_family_label, "integrated_gradients", labels, baseline_scores, rng, serialize_scores=True, score_column="ig_scores_json"))
        if "gradient_x_input" in methods:
            mtl_scores = baseline_notebook_utils.compute_gradient_x_input_scores(model, tokenizer, sequence, DEVICE)
            probe_rows.extend(mtl_epitope_notebook_utils.build_probe_rows_with_label_scrambling(base, output_paths.mtl_family_label, "gradient_x_input", labels, mtl_scores, rng, serialize_scores=True, score_column="gradient_x_input_scores_json"))
            baseline_scores = baseline_notebook_utils.compute_gradient_x_input_scores(baseline_model, tokenizer, sequence, DEVICE)
            baseline_rows.extend(mtl_epitope_notebook_utils.build_probe_rows_with_label_scrambling(base, output_paths.baseline_family_label, "gradient_x_input", labels, baseline_scores, rng, serialize_scores=True, score_column="gradient_x_input_scores_json"))
        if "smoothgrad_ig" in methods:
            mtl_scores = baseline_notebook_utils.compute_smoothgrad_ig_scores(model, tokenizer, sequence, DEVICE, steps=IG_STEPS, n_samples=SMOOTHGRAD_IG_SAMPLES, noise_std=SMOOTHGRAD_IG_NOISE_STD, internal_batch_size=IG_INTERNAL_BATCH_SIZE)
            probe_rows.extend(mtl_epitope_notebook_utils.build_probe_rows_with_label_scrambling(base, output_paths.mtl_family_label, "smoothgrad_ig", labels, mtl_scores, rng, serialize_scores=True, score_column="smoothgrad_ig_scores_json"))
            baseline_scores = baseline_notebook_utils.compute_smoothgrad_ig_scores(baseline_model, tokenizer, sequence, DEVICE, steps=IG_STEPS, n_samples=SMOOTHGRAD_IG_SAMPLES, noise_std=SMOOTHGRAD_IG_NOISE_STD, internal_batch_size=IG_INTERNAL_BATCH_SIZE)
            baseline_rows.extend(mtl_epitope_notebook_utils.build_probe_rows_with_label_scrambling(base, output_paths.baseline_family_label, "smoothgrad_ig", labels, baseline_scores, rng, serialize_scores=True, score_column="smoothgrad_ig_scores_json"))
    return pd.DataFrame(probe_rows), pd.DataFrame(baseline_rows), output_paths


def repair_selected_probe_methods() -> None:
    patch_live_attribution_helpers()
    requested_methods = set(REPAIR_METHODS) & enabled_gradient_methods()
    if not requested_methods:
        print("No enabled repair methods selected. Nothing to do.")
        return
    target_families = set(REPAIR_FAMILY_KEYS)
    print(f"Repairing methods: {sorted(requested_methods)}")
    print(f"Target families: {sorted(target_families)}")

    if "baseline" in target_families and baseline_spec["checkpoint_exists"]:
        existing_baseline = pd.read_csv(baseline_spec["probe_rows_path"]) if Path(baseline_spec["probe_rows_path"]).exists() else pd.DataFrame()
        new_baseline = recompute_baseline_gradient_methods(baseline_spec, requested_methods)
        merged_baseline = merge_probe_rows(existing_baseline, new_baseline, requested_methods)
        merged_baseline.to_csv(baseline_spec["probe_rows_path"], index=False)
        save_registry_probe_summary(merged_baseline, baseline_spec["summary_path"], selected_methods_for_spec(baseline_spec))
        print(f"Updated baseline probe rows: {baseline_spec['probe_rows_path']}")

    for spec in supported_registry:
        if spec["family_key"] == "baseline" or spec["family_key"] not in target_families:
            continue
        if not spec["checkpoint_exists"]:
            print(f"Skipping missing checkpoint for repair: {spec['checkpoint_name']}")
            continue
        if spec["model_kind"] == "mtl":
            probe_path = spec["probe_rows_path"]
            output_paths = build_output_paths_for_supported_mtl(
                family_key=spec["family_key"],
                display_label=spec["display_label"],
                models_dir=MODELS_DIR,
                results_dir=RESULTS_DIR,
                baseline_checkpoint_path=baseline_spec["checkpoint_path"],
                baseline_summary_path=baseline_spec["summary_path"],
            )
            existing_probe = pd.read_csv(probe_path) if Path(probe_path).exists() else pd.DataFrame()
            existing_baseline = pd.read_csv(output_paths.baseline_probe_rows_path) if Path(output_paths.baseline_probe_rows_path).exists() else pd.DataFrame()
            new_probe, new_baseline, output_paths = recompute_mtl_gradient_methods(spec, requested_methods)
            merged_probe = merge_probe_rows(existing_probe, new_probe, requested_methods)
            merged_baseline = merge_probe_rows(existing_baseline, new_baseline, requested_methods)
            merged_combined = mtl_epitope_notebook_utils.ensure_label_variant_column(pd.concat([merged_baseline, merged_probe], ignore_index=True))
            mtl_epitope_notebook_utils.validate_unique_probe_rows(merged_probe)
            mtl_epitope_notebook_utils.validate_unique_probe_rows(merged_baseline)
            mtl_epitope_notebook_utils.validate_unique_probe_rows(merged_combined)
            merged_probe.to_csv(output_paths.probe_rows_path, index=False)
            merged_baseline.to_csv(output_paths.baseline_probe_rows_path, index=False)
            merged_combined.to_csv(output_paths.combined_probe_rows_path, index=False)
            summarize_probe_outputs(merged_probe, merged_baseline, output_paths)
            print(f"Updated MTL probe rows: {output_paths.probe_rows_path}")
            continue
        if spec["model_kind"] == "deep_plant" and "integrated_gradients" in requested_methods:
            existing_probe = pd.read_csv(spec["probe_rows_path"]) if Path(spec["probe_rows_path"]).exists() else pd.DataFrame()
            deep_plant_model, deep_plant_metadata = load_deep_plant_allergy_checkpoint(spec["checkpoint_path"], DEVICE)
            deep_plant_hf_model_name = deep_plant_metadata.get("hf_model_name")
            deep_plant_tokenizer = build_deep_plant_tokenizer(deep_plant_hf_model_name)
            deep_plant_embedding_model = build_embedding_model(deep_plant_hf_model_name, device=DEVICE)
            rows = []
            desc = f"Repairing [{spec['display_label']}]"
            for _, row in tqdm(epitope_probe_df.iterrows(), total=len(epitope_probe_df), desc=desc):
                sequence = row["sequence"]
                labels = row["epitope_label"]
                scores = deep_plant_allergy_utils.compute_integrated_gradients(
                    deep_plant_model, deep_plant_embedding_model, deep_plant_tokenizer, sequence, DEVICE, steps=IG_STEPS, normalize=False, internal_batch_size=IG_INTERNAL_BATCH_SIZE,
                )
                rows.append({
                    "accession": str(row["accession"]),
                    "seq_len": row["seq_len"],
                    "epitope_density": row["epitope_density"],
                    "n_epitope_residues": row["n_epitope_residues"],
                    "model_family": spec["display_label"],
                    "method": "integrated_gradients",
                    "ig_scores_json": deep_plant_allergy_utils.serialize_score_array(scores),
                    **deep_plant_allergy_utils.compute_probe_metrics(labels, scores),
                })
            new_probe = pd.DataFrame(rows)
            merged_probe = merge_probe_rows(existing_probe, new_probe, {"integrated_gradients"})
            merged_probe.to_csv(spec["probe_rows_path"], index=False)
            save_registry_probe_summary(merged_probe, spec["summary_path"], selected_methods_for_spec(spec))
            print(f"Updated DeepPlantAllergy probe rows: {spec['probe_rows_path']}")


repair_selected_probe_methods()


In [16]:
import inspect

generation_records = []
deep_plant_embedding_model = None
deep_plant_tokenizer = None
baseline_probe_df = None

def _available_methods_from_rows(probe_rows_path: Path) -> str:
    if not Path(probe_rows_path).exists():
        return ""
    probe_df = pd.read_csv(probe_rows_path, usecols=["method"])
    methods = sorted(str(value) for value in probe_df["method"].dropna().unique())
    return ", ".join(methods)

def _append_record(spec: dict, status: str, probe_rows_path: Path | None = None) -> None:
    generation_records.append(
        {
            "checkpoint_name": spec["checkpoint_name"],
            "family_key": spec["family_key"],
            "display_label": spec["display_label"],
            "status": status,
            "available_methods": _available_methods_from_rows(probe_rows_path or spec["probe_rows_path"]),
            "probe_rows_path": str(probe_rows_path or spec["probe_rows_path"]),
        }
    )

def _supports_progress_label(fn) -> bool:
    return "progress_label" in inspect.signature(fn).parameters

def _supports_enabled_methods(fn) -> bool:
    return "enabled_methods" in inspect.signature(fn).parameters

def _selected_methods_for_spec(spec: dict) -> list[str]:
    methods = []
    for method in spec["supported_methods"]:
        if method == "integrated_gradients" and not INCLUDE_INTEGRATED_GRADIENTS:
            continue
        if method == "gradient_x_input" and not INCLUDE_GRADIENT_X_INPUT:
            continue
        if method == "smoothgrad_ig" and not INCLUDE_SMOOTHGRAD_IG:
            continue
        methods.append(method)
    return methods

BASELINE_SUPPORTS_PROGRESS = _supports_progress_label(run_baseline_probe_suite)
DEEP_PLANT_SUPPORTS_PROGRESS = _supports_progress_label(run_deep_plant_probe_suite)
MTL_SUPPORTS_PROGRESS = _supports_progress_label(run_probe_suite)
BASELINE_SUPPORTS_ENABLED_METHODS = _supports_enabled_methods(run_baseline_probe_suite)
DEEP_PLANT_SUPPORTS_ENABLED_METHODS = _supports_enabled_methods(run_deep_plant_probe_suite)
MTL_SUPPORTS_ENABLED_METHODS = _supports_enabled_methods(run_probe_suite)

baseline_spec = next(spec for spec in supported_registry if spec["family_key"] == "baseline")
if (not INCLUDE_GRADIENT_X_INPUT or not INCLUDE_SMOOTHGRAD_IG or not INCLUDE_INTEGRATED_GRADIENTS) and not MTL_SUPPORTS_ENABLED_METHODS:
    print("Current MTL helper does not support method-level filtering in this runtime; use the standalone repair cell for selective reruns.")
if baseline_spec["probe_rows_exists"] and not OVERWRITE_EXISTING:
    baseline_probe_df = pd.read_csv(baseline_spec["probe_rows_path"])
    baseline_probe_df["model_family"] = baseline_spec["display_label"]
    print(f"Reusing baseline probe rows: {baseline_spec['probe_rows_path']}")
    _append_record(baseline_spec, "reused")
elif baseline_spec["checkpoint_exists"]:
    baseline_model, _ = load_baseline_checkpoint(baseline_spec["checkpoint_path"], DEVICE)
    baseline_kwargs = dict(
        model=baseline_model,
        tokenizer=tokenizer,
        eval_df=epitope_probe_df,
        device=DEVICE,
        ig_steps=IG_STEPS,
        n_random_draws=N_RANDOM_DRAWS,
        ig_internal_batch_size=IG_INTERNAL_BATCH_SIZE,
        smoothgrad_ig_samples=SMOOTHGRAD_IG_SAMPLES,
        smoothgrad_ig_noise_std=SMOOTHGRAD_IG_NOISE_STD,
        include_shuffled_mean=False,
    )
    if BASELINE_SUPPORTS_PROGRESS:
        baseline_kwargs["progress_label"] = f"Evaluating proteins [{baseline_spec['display_label']}]"
    else:
        print(f"Evaluating proteins [{baseline_spec['display_label']}]")
    if BASELINE_SUPPORTS_ENABLED_METHODS:
        baseline_kwargs["enabled_methods"] = set(_selected_methods_for_spec(baseline_spec))
    baseline_probe_df = run_baseline_probe_suite(**baseline_kwargs)
    baseline_probe_df["model_family"] = baseline_spec["display_label"]
    baseline_probe_df.to_csv(baseline_spec["probe_rows_path"], index=False)
    save_registry_probe_summary(
        baseline_probe_df,
        baseline_spec["summary_path"],
        baseline_spec["supported_methods"],
    )
    print(f"Saved baseline probe rows to: {baseline_spec['probe_rows_path']}")
    _append_record(baseline_spec, "generated")
else:
    print(f"Skipping missing supported checkpoint: {baseline_spec['checkpoint_name']}")
    _append_record(baseline_spec, "skipped_missing")

for spec in supported_registry:
    if spec["family_key"] == "baseline":
        continue
    if spec["probe_rows_exists"] and not OVERWRITE_EXISTING:
        print(f"Reusing probe rows for {spec['display_label']}: {spec['probe_rows_path']}")
        _append_record(spec, "reused")
        continue
    if not spec["checkpoint_exists"]:
        print(f"Skipping missing supported checkpoint: {spec['checkpoint_name']}")
        _append_record(spec, "skipped_missing")
        continue
    if spec["model_kind"] == "deep_plant":
        deep_plant_model, deep_plant_metadata = load_deep_plant_allergy_checkpoint(spec["checkpoint_path"], DEVICE)
        deep_plant_hf_model_name = deep_plant_metadata.get("hf_model_name")
        if deep_plant_embedding_model is None:
            deep_plant_tokenizer = build_deep_plant_tokenizer(deep_plant_hf_model_name)
            deep_plant_embedding_model = build_embedding_model(deep_plant_hf_model_name, device=DEVICE)
        deep_plant_kwargs = dict(
            model=deep_plant_model,
            embedding_model=deep_plant_embedding_model,
            tokenizer=deep_plant_tokenizer,
            eval_df=epitope_probe_df,
            device=DEVICE,
            ig_steps=IG_STEPS,
            n_random_draws=N_RANDOM_DRAWS,
        )
        if DEEP_PLANT_SUPPORTS_PROGRESS:
            deep_plant_kwargs["progress_label"] = f"Evaluating proteins [{spec['display_label']}]"
        else:
            print(f"Evaluating proteins [{spec['display_label']}]")
        if DEEP_PLANT_SUPPORTS_ENABLED_METHODS:
            deep_plant_kwargs["enabled_methods"] = set(_selected_methods_for_spec(spec))
        probe_df = run_deep_plant_probe_suite(**deep_plant_kwargs)
        probe_df["model_family"] = spec["display_label"]
        probe_df.to_csv(spec["probe_rows_path"], index=False)
        save_registry_probe_summary(
            probe_df,
            spec["summary_path"],
            _selected_methods_for_spec(spec),
        )
        _append_record(spec, "generated")
        continue
    output_paths = build_output_paths_for_supported_mtl(
        family_key=spec["family_key"],
        display_label=spec["display_label"],
        models_dir=MODELS_DIR,
        results_dir=RESULTS_DIR,
        baseline_checkpoint_path=baseline_spec["checkpoint_path"],
        baseline_summary_path=baseline_spec["summary_path"],
    )
    model, _ = load_mtl_checkpoint(
        spec["checkpoint_path"],
        DEVICE,
        model_name=HF_MODEL_NAME,
        hidden_dim=HIDDEN_DIM,
        dropout=DROPOUT,
    )
    mtl_kwargs = dict(
        model=model,
        tokenizer=tokenizer,
        epitope_probe_df=epitope_probe_df,
        baseline_checkpoint_path=baseline_spec["checkpoint_path"],
        device=DEVICE,
        hidden_dim=HIDDEN_DIM,
        dropout=DROPOUT,
        output_paths=output_paths,
        ig_steps=IG_STEPS,
        n_random_draws=N_RANDOM_DRAWS,
        ig_internal_batch_size=IG_INTERNAL_BATCH_SIZE,
        resume=False,
        precomputed_baseline_probe_df=baseline_probe_df,
    )
    if MTL_SUPPORTS_PROGRESS:
        mtl_kwargs["progress_label"] = f"Probing splitB [{spec['display_label']}]"
    else:
        print(f"Probing splitB [{spec['display_label']}]")
    if MTL_SUPPORTS_ENABLED_METHODS:
        mtl_kwargs["enabled_methods"] = set(_selected_methods_for_spec(spec))
    probe_outputs = run_probe_suite(**mtl_kwargs)
    summarize_probe_outputs(
        probe_df=probe_outputs["probe_df"],
        baseline_probe_df=probe_outputs["baseline_probe_df"],
        output_paths=output_paths,
    )
    _append_record(spec, "generated", probe_rows_path=output_paths.probe_rows_path)

processed_checkpoint_names = {spec["checkpoint_name"] for spec in supported_registry}
for checkpoint_path in sorted(MODELS_DIR.glob("*.pt")):
    if checkpoint_path.name in processed_checkpoint_names:
        continue
    if checkpoint_path.name in retired_checkpoint_names:
        print(f"Skipping retired baseline checkpoint: {checkpoint_path.name}")
        generation_records.append(
            {
                "checkpoint_name": checkpoint_path.name,
                "family_key": "retired",
                "display_label": "Retired baseline",
                "status": "skipped_retired",
                "available_methods": "",
                "probe_rows_path": "",
            }
        )
        continue
    print(f"Skipping unknown checkpoint: {checkpoint_path.name}")
    generation_records.append(
        {
            "checkpoint_name": checkpoint_path.name,
            "family_key": "unknown",
            "display_label": "Unsupported checkpoint",
            "status": "skipped_unknown",
            "available_methods": "",
            "probe_rows_path": "",
        }
    )

generation_df = pd.DataFrame(generation_records)
generation_df


Some weights of EsmModel were not initialized from the model checkpoint at /root/.cache/huggingface/hub/models--facebook--esm2_t6_8M_UR50D/snapshots/c731040fcd8d73dceaa04b0a8e6329b345b0f5df and are newly initialized: ['esm.pooler.dense.bias', 'esm.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Evaluating proteins [Frozen ESM-2]


Evaluating proteins:   0%|          | 0/61 [00:00<?, ?it/s]

Some weights of EsmModel were not initialized from the model checkpoint at /root/.cache/huggingface/hub/models--facebook--esm2_t6_8M_UR50D/snapshots/c731040fcd8d73dceaa04b0a8e6329b345b0f5df and are newly initialized: ['esm.pooler.dense.bias', 'esm.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Saved baseline probe rows to: /content/drive/MyDrive/XAllergen2.0/results/probing/rows/baseline_probing_rows.csv
Probing splitB [MTL ESM-2]
Precomputed baseline probe rows are missing methods or label variants (methods=[], label_variants=['scrambled']); recomputing baseline probes.


...

Some weights of EsmModel were not initialized from the model checkpoint at /root/.cache/huggingface/hub/models--facebook--esm2_t6_8M_UR50D/snapshots/c731040fcd8d73dceaa04b0a8e6329b345b0f5df and are newly initialized: ['esm.pooler.dense.bias', 'esm.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of EsmModel were not initialized from the model checkpoint at /root/.cache/huggingface/hub/models--facebook--esm2_t6_8M_UR50D/snapshots/c731040fcd8d73dceaa04b0a8e6329b345b0f5df and are newly initialized: ['esm.pooler.dense.bias', 'esm.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Probing splitB [MTL ESM-2 top-1]
Precomputed baseline probe rows are missing methods or label variants (methods=[], label_variants=['scrambled']); recomputing baseline probes.
Integrated Gradients device: cuda
IG_STEPS: 50
IG internal_batch_size: 64
SmoothGrad-IG samples: 10
SmoothGrad-IG noise_std: 0.05


Probing splitB:   0%|          | 0/61 [00:00<?, ?it/s]

Saved MTL row-wise probe metrics to: /content/drive/MyDrive/XAllergen2.0/results/probing/rows/mtl_top1_unfrozen_probing_rows.csv
Saved baseline row-wise probe metrics to: /content/drive/MyDrive/XAllergen2.0/results/probing/rows/baseline_probing_rows_top1_unfrozen.csv
Saved combined row-wise probe metrics to: /content/drive/MyDrive/XAllergen2.0/results/probing/rows/mtl_top1_unfrozen_vs_baseline_probing_rows.csv
   model_family               method label_variant              method_category  auroc_mean  auroc_ci_low  auroc_ci_high  auprc_mean  auprc_ci_low  auprc_ci_high  precision_at_k_mean  precision_at_k_ci_low  precision_at_k_ci_high  n_proteins
MTL ESM-2 top-1          random_mean      original                Null baseline      0.5007        0.4993         0.5021      0.2673        0.2176         0.3196               0.2501                 0.2000                  0.3027          61
MTL ESM-2 top-1          random_mean     scrambled                Null baseline      0.4988        0.4

,checkpoint_name,family_key,display_label,status,available_methods,probe_rows_path
0,baseline_frozen_esm2.pt,baseline,Frozen ESM-2,generated,"attention_weights, gradient_x_input, integrate...",/content/drive/MyDrive/XAllergen2.0/results/pr...
1,mtl_frozen_esm2_epitope.pt,mtl_frozen,MTL ESM-2,generated,"attention_weights, gradient_x_input, integrate...",/content/drive/MyDrive/XAllergen2.0/results/pr...
2,deep_plant_allergy_benchmark.pt,deep_plant_benchmark,DeepPlantAllergy,generated,"attention_weights, random_mean, shuffled_mean",/content/drive/MyDrive/XAllergen2.0/results/pr...
3,mtl_top1_unfrozen_esm2_epitope.pt,mtl_top1_unfrozen,MTL ESM-2 top-1,generated,"attention_weights, gradient_x_input, integrate...",/content/drive/MyDrive/XAllergen2.0/results/pr...
4,baseline_top1_unfrozen_esm2.pt,retired,Retired baseline,skipped_retired,,


## Generation Summary

The final summary table reports the checkpoint name, registry family key, publication label, generation status, methods present in the saved row artifact, and the probe-row output path.


In [ ]:
summary_columns = [
    "checkpoint_name",
    "family_key",
    "display_label",
    "status",
    "available_methods",
    "probe_rows_path",
]
print(generation_df[summary_columns].to_string(index=False))


In [ ]:
# %% [markdown]
# ## Optional: Disconnect Colab Runtime After Successful Completion

# %%
AUTO_DISCONNECT_COLAB = True

if AUTO_DISCONNECT_COLAB and IN_COLAB:
    print("Notebook finished successfully. Disconnecting and deleting Colab runtime to stop GPU usage.")
    from google.colab import runtime  # type: ignore
    runtime.unassign()